In [1]:
import pandas as pd
import numpy as np  

In [2]:
df = pd.read_csv('Nonlife.csv')
df = df.sort_values(by='Date', ascending=True)
print(df.head())

           Symbol        Date      Open      High       Low     Close  \
1157  NONLIFEINSU  2021-03-31  13669.61  13722.74  13464.81  13674.44   
1156  NONLIFEINSU  2021-04-01  14036.45  14042.50  13668.81  13699.45   
1155  NONLIFEINSU  2021-04-04  13911.52  14327.62  13895.67  14243.43   
1154  NONLIFEINSU  2021-04-05  14420.40  14492.81  14016.25  14136.82   
1153  NONLIFEINSU  2021-04-06  14226.88  14226.88  13843.06  13849.06   

     Percent Change Volume Turn Over  
1157         0.00 %      -         -  
1156         0.00 %      -         -  
1155         0.00 %      -         -  
1154         0.00 %      -         -  
1153         0.00 %      -         -  


In [3]:
df['tp'] = (df['Close']+df['High']+df['Low'])/3

In [4]:
# Clean thousands separators before numeric conversion
df['Volume'] = pd.to_numeric(df['Volume'].astype(str).str.replace(',', '', regex=False), errors='coerce')
df['rmf'] = df['tp'] * df['Volume']

In [5]:
#Find postive and negative money flow
df['tp_diff'] = df['tp'].diff()
df['Postive_tp']= df['tp_diff'] > 0
df['Negative_tp']= df['tp_diff'] < 0


In [6]:
df['pos_flow'] = df['rmf'].where(df['Postive_tp'], 0)
df['neg_flow'] = df['rmf'].where(df['Negative_tp'], 0)

pos_sum = df['pos_flow'].rolling(14).sum()
neg_sum = df['neg_flow'].rolling(14).sum()

mfr = pos_sum / neg_sum
df['mfi'] = 100 - (100 / (1 + mfr))

In [7]:
df.tail()

,Symbol,Date,Open,High,Low,Close,Percent Change,Volume,Turn Over,tp,rmf,tp_diff,Postive_tp,Negative_tp,pos_flow,neg_flow,mfi
4,NONLIFEINSU,2026-03-24,12005.57,12027.13,11879.23,11945.64,-0.38 %,163311995.0,-,11950.666667,1.951687e+12,-19.723333,False,True,0.0,1.951687e+12,71.503859
3,NONLIFEINSU,2026-03-25,11958.99,11962.17,11797.16,11808.08,-1.15 %,222316485.5,-,11855.803333,2.635741e+12,-94.863333,False,True,0.0,2.635741e+12,63.469548
2,NONLIFEINSU,2026-03-26,11808.09,11863.83,11708.94,11858.96,0.43 %,138676836.2,-,11810.576667,1.637853e+12,-45.226667,False,True,0.0,1.637853e+12,58.612360
1,NONLIFEINSU,2026-03-29,11873.12,11886.20,11576.87,11594.31,-2.23 %,234775431.4,-,11685.793333,2.743537e+12,-124.783333,False,True,0.0,2.743537e+12,51.784448
0,NONLIFEINSU,2026-03-30,11594.31,11595.48,11337.71,11382.73,-1.82 %,189004398.4,-,11438.640000,2.161953e+12,-247.153333,False,True,0.0,2.161953e+12,48.275694


In [8]:
# Prepare data for plotting: convert date to datetime
df_plot = df.copy()
df_plot['Date'] = pd.to_datetime(df_plot['Date'])

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots with 2 rows and shared x-axis
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    row_heights=[0.7, 0.3],
    subplot_titles=("Candlestick Chart", "Money Flow Index (MFI)")
)

# Convert date to string to use as categorical axis (preserves gaps)
df_plot['Date_str'] = df_plot['Date'].astype(str)

# Add candlestick chart to the first subplot
fig.add_trace(
    go.Candlestick(
        x=df_plot['Date_str'],
        open=df_plot['Open'],
        high=df_plot['High'],
        low=df_plot['Low'],
        close=df_plot['Close'],
        name='OHLC',
        increasing_line_color='green',
        decreasing_line_color='red',
    ),
    row=1, col=1
)

# Add MFI line chart to the second subplot
fig.add_trace(
    go.Scatter(
        x=df_plot['Date_str'],
        y=df_plot['mfi'],
        mode='lines',
        name='MFI',
        line=dict(color='blue', width=2),
        fill='tozeroy',
        fillcolor='rgba(0, 0, 255, 0.2)'
    ),
    row=2, col=1
)

# Add horizontal line at 70 and 30 for overbought/oversold
fig.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought", row=2)
fig.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold", row=2)

# Update layout
fig.update_layout(
    height=700,
    title_text="Trading View with MFI Indicator",
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    template='plotly_white'
)

# Update y-axes labels
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="MFI", row=2, col=1)

fig.show()

In [10]:
# Swing detection function
def find_swings(series, lookback=10):
    highs, lows = [], []
    for i in range(lookback, len(series) - lookback):
        window = series.iloc[i-lookback: lookback + i +1]
        current = float(series.iloc[i])
        if current == float(window.max()):
            highs.append(i)
        if current == float(window.min()):
            lows.append(i)
    return highs, lows

# Find swing highs and lows for price
high_idx, low_idx = find_swings(df_plot["Close"])

# Divergence detection
regular_bullish, regular_bearish = [], []
hidden_bullish, hidden_bearish = [], []

# Regular Bullish: Price LL, MFI HL
for i in range(1, len(low_idx)):
    p1, p2 = low_idx[i - 1], low_idx[i]
    if df_plot["Close"].iloc[p2] < df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] > df_plot["mfi"].iloc[p1]:
        regular_bullish.append((p1, p2))

# Regular Bearish: Price HH, MFI LH
for i in range(1, len(high_idx)):
    p1, p2 = high_idx[i - 1], high_idx[i]
    if df_plot["Close"].iloc[p2] > df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] < df_plot["mfi"].iloc[p1]:
        regular_bearish.append((p1, p2))

# Hidden Bullish: Price HL, MFI LL
for i in range(1, len(low_idx)):
    p1, p2 = low_idx[i-1], low_idx[i]
    if df_plot["Close"].iloc[p2] > df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] < df_plot["mfi"].iloc[p1]:
        hidden_bullish.append((p1, p2))

# Hidden Bearish: Price LH, MFI HH
for i in range(1, len(high_idx)):
    p1, p2 = high_idx[i-1], high_idx[i]
    if df_plot["Close"].iloc[p2] < df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] > df_plot["mfi"].iloc[p1]:
        hidden_bearish.append((p1, p2))

print("Regular Bullish Divergences:", regular_bullish)
print("Regular Bearish Divergences:", regular_bearish)
print("Hidden Bullish Divergences:", hidden_bullish)
print("Hidden Bearish Divergences:", hidden_bearish)

Regular Bullish Divergences: [(65, 132), (224, 294), (368, 380), (460, 482), (565, 585), (691, 723), (994, 1060)]
Regular Bearish Divergences: [(303, 325), (428, 534), (534, 553), (678, 697), (948, 961)]
Hidden Bullish Divergences: [(417, 460), (507, 565), (822, 841), (1079, 1101)]
Hidden Bearish Divergences: [(98, 142), (195, 235), (386, 404), (568, 593), (1038, 1065)]


In [11]:
# Plot divergence lines on the chart
fig_div = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    row_heights=[0.7, 0.3],
    subplot_titles=("Candlestick Chart with Divergences", "Money Flow Index (MFI)")
)

# Add candlestick chart
fig_div.add_trace(
    go.Candlestick(
        x=df_plot['Date_str'],
        open=df_plot['Open'],
        high=df_plot['High'],
        low=df_plot['Low'],
        close=df_plot['Close'],
        name='OHLC',
        increasing_line_color='green',
        decreasing_line_color='red',
    ),
    row=1, col=1
)

# Add MFI
fig_div.add_trace(
    go.Scatter(
        x=df_plot['Date_str'],
        y=df_plot['mfi'],
        mode='lines',
        name='MFI',
        line=dict(color='blue', width=2)
      
    ),
    row=2, col=1
)

# Regular Bullish on PRICE
for p1, p2 in regular_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='green', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bullish<extra></extra>'
        ),
        row=1, col=1
    )

# Regular Bearish on PRICE
for p1, p2 in regular_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bearish<extra></extra>'
        ),
        row=1, col=1
    )

# Hidden Bullish on PRICE
for p1, p2 in hidden_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='lime', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Hidden Bullish<extra></extra>'
        ),
        row=1, col=1
    )

# Hidden Bearish on PRICE
for p1, p2 in hidden_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='orange', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Hidden Bearish<extra></extra>'
        ),
        row=1, col=1
    )

# Regular Bullish on MFI
for p1, p2 in regular_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='green', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bullish<extra></extra>'
        ),
        row=2, col=1
    )

# Regular Bearish on MFI
for p1, p2 in regular_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bearish<extra></extra>'
        ),
        row=2, col=1
    )

# Hidden Bullish on MFI
for p1, p2 in hidden_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='lime', dash='dot', width=2),
            showlegend=False,
            hovertemplate='Hidden Bullish<extra></extra>'
        ),
        row=2, col=1
    )

# Hidden Bearish on MFI
for p1, p2 in hidden_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='orange', dash='dot', width=2),
            showlegend=False,
            hovertemplate='Hidden Bearish<extra></extra>'
        ),
        row=2, col=1
    )

# Add overbought/oversold lines (faded)
fig_div.add_hline(y=70, line_dash="dash", line_color="rgba(128, 128, 128, 0.2)", row=2)
fig_div.add_hline(y=30, line_dash="dash", line_color="rgba(128, 128, 128, 0.2)", row=2)

# Add legend entries manually
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='green', dash='dash', width=2), 
                         name='Regular Bullish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='red', dash='dash', width=2), 
                         name='Regular Bearish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='lime', dash='dot', width=2), 
                         name='Hidden Bullish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='orange', dash='dot', width=2), 
                         name='Hidden Bearish'), row=1, col=1)

# Update layout
fig_div.update_layout(
    height=700,
    width=1400,
    title_text="Trading View with MFI Divergences",
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0, y=1)
)

fig_div.update_yaxes(title_text="Price", row=1, col=1)
fig_div.update_yaxes(title_text="MFI", row=2, col=1)

# Print summary
print(f"Regular Bullish Divergences: {len(regular_bullish)}")
print(f"Regular Bearish Divergences: {len(regular_bearish)}")
print(f"Hidden Bullish Divergences: {len(hidden_bullish)}")
print(f"Hidden Bearish Divergences: {len(hidden_bearish)}")

fig_div.show()

Regular Bullish Divergences: 7
Regular Bearish Divergences: 5
Hidden Bullish Divergences: 4
Hidden Bearish Divergences: 5


In [12]:
y_r_bullish = [y for _, y in regular_bullish]
y_r_bearish = [y for _, y in regular_bearish]
y_h_bullish = [y for _, y in hidden_bullish]
y_h_bearish = [y for _, y in hidden_bearish]

In [13]:
candle_intervals= [5,10,20,40,60]
results=[]

In [14]:
all_data=[]
for index in y_r_bullish:
    all_data.append((index, "Regular Bullish"))
for index in y_r_bearish:
    all_data.append((index, "Regular Bearish"))
for index in y_h_bullish:
    all_data.append((index, "Hidden Bullish"))
for index in y_h_bearish:
    all_data.append((index, "Hidden Bearish"))
    

In [15]:
symbol= df["Symbol"][0]
symbol

'NONLIFEINSU'

In [16]:
for i,pattern in all_data:
    if i+max(candle_intervals)>= len(df):
        continue
    for j in candle_intervals:
        price_now = df["Close"].iloc[i]
        price_future = df["Close"].iloc[i+j]

        pc= (price_future-price_now)/price_now *100

        results.append({
            "Sector": symbol,
            "Pattern":pattern,
            "Interval":j,
            "Start Index":i,
            "EndIndex": i+j,
            "Price Now":price_now,
            "Price After Interval":price_future,
            "Percentage Change": pc
        })
        

In [17]:
results = pd.DataFrame(results)

In [18]:
results

,Sector,Pattern,Interval,Start Index,EndIndex,Price Now,Price After Interval,Percentage Change
0,NONLIFEINSU,Regular Bullish,5,132,137,11255.10,12310.97,9.381258
1,NONLIFEINSU,Regular Bullish,10,132,142,11255.10,13205.86,17.332232
2,NONLIFEINSU,Regular Bullish,20,132,152,11255.10,12351.06,9.737452
3,NONLIFEINSU,Regular Bullish,40,132,172,11255.10,10028.09,-10.901813
4,NONLIFEINSU,Regular Bullish,60,132,192,11255.10,13207.00,17.342360
...,...,...,...,...,...,...,...,...
95,NONLIFEINSU,Hidden Bearish,5,1065,1070,11036.82,10671.64,-3.308743
96,NONLIFEINSU,Hidden Bearish,10,1065,1075,11036.82,10541.60,-4.486981
97,NONLIFEINSU,Hidden Bearish,20,1065,1085,11036.82,11026.69,-0.091784
98,NONLIFEINSU,Hidden Bearish,40,1065,1105,11036.82,10994.86,-0.380182


In [19]:
results.to_excel(f"{symbol}_MFI_Divergence.xlsx",index=False)